In [1]:
import hashlib
import secrets
import smtplib
from email.mime.text import MIMEText
import csv

In [4]:
import pandas as pd

file = r"C:\Users\Rajat\Downloads\Existing list of M.c studs till 17.04.26.xlsx"

df = pd.read_excel(file, sheet_name=0)

print(df.columns.tolist())

['Sl. No.', 'NISER Roll No', 'Full Name', 'Date of Birth', 'Gender']


In [14]:

master_file = r"C:\Users\Rajat\Downloads\Existing list of M.c studs till 17.04.26.xlsx"

In [15]:
import pandas as pd

email_file = r"C:\Users\Rajat\Downloads\Consolidated Email List.xlsx"

df = pd.read_excel(file, sheet_name=0)

print(df.columns.tolist())

['S No', 'Name ', 'Email', 'Full Email']


In [16]:
import pandas as pd
import re
from rapidfuzz import process, fuzz

In [31]:
import pandas as pd
import math

# =====================================================
# FILE PATH
# =====================================================
input_file = r"C:\Users\Rajat\Downloads\Consolidated Email List.xlsx"
output_file = r"C:\Users\Rajat\Downloads\Divided_Into_4_Batches.xlsx"

# =====================================================
# READ FILE
# =====================================================
df = pd.read_excel(input_file)

# Clean column names
df.columns = [str(c).strip() for c in df.columns]

# =====================================================
# DETECT NAME + EMAIL COLUMNS
# =====================================================
name_col = None
email_col = None

for col in df.columns:
    c = col.lower()

    if name_col is None and "name" in c:
        name_col = col

    if email_col is None and ("full email" in c):
        email_col = col

if name_col is None or email_col is None:
    raise Exception("Could not detect Name / Email columns")

# Keep only required columns
df = df[[name_col, email_col]].copy()
df.columns = ["Full Name", "Full Email"]

# Remove blanks
df = df.dropna()

# Remove duplicates
df = df.drop_duplicates(subset=["Full Email"])

# Reset index
df = df.reset_index(drop=True)

# =====================================================
# DIVIDE INTO 4 EQUAL BATCHES
# =====================================================
total = len(df)
chunk = math.ceil(total / 4)

writer = pd.ExcelWriter(output_file, engine="openpyxl")

for i in range(4):
    start = i * chunk
    end = min((i + 1) * chunk, total)

    batch_df = df.iloc[start:end].copy()

    # Add Serial No
    batch_df.insert(0, "S No", range(1, len(batch_df) + 1))

    sheet_name = f"Batch_{i+1}"

    batch_df.to_excel(writer, sheet_name=sheet_name, index=False)

writer.close()

# =====================================================
# SUMMARY
# =====================================================
print("✅ Done.")
print("Saved:", output_file)
print("Total Emails:", total)
print("Each divided into 4 sheets.")

✅ Done.
Saved: C:\Users\Rajat\Downloads\Divided_Into_4_Batches.xlsx
Total Emails: 1438
Each divided into 4 sheets.


In [9]:
import dns.resolver
import socket

domain = "niser.ac.in"

def get_mail_host(domain):
    try:
        answers = dns.resolver.resolve(domain, 'MX')
        records = sorted(answers, key=lambda r: r.preference)
        return str(records[0].exchange).rstrip('.')
    except:
        print("No MX found. Trying A record...")
        try:
            answers = dns.resolver.resolve(domain, 'A')
            return domain
        except:
            return None

host = get_mail_host(domain)
print("Mail Host:", host)


No MX found. Trying A record...
Mail Host: niser.ac.in


In [10]:
import smtplib

host = "niser.ac.in"

try:
    server = smtplib.SMTP(host, 25, timeout=15)
    server.ehlo()
    print("Connected successfully")
    print(server.noop())
    server.quit()

except Exception as e:
    print("Failed:", e)

Failed: timed out


In [3]:
import pandas as pd
import secrets
import hashlib
import smtplib
import time

from email.mime.text import MIMEText


# =====================================================
# EMAIL LOGIN
# =====================================================
EMAIL = "rajatshuvra.saha@niser.ac.in"
PASSWORD = "jfij ehfa hrjo shbx"

# =====================================================
# EXCEL FILE WITH 4 SHEETS
# =====================================================
excel_file = r"C:\Users\Rajat\Downloads\Testing_my_Names.xlsx"

# =====================================================
# SETTINGS
# =====================================================
KEY_LENGTH = 5
DELAY = 1   # seconds between mails

# =====================================================
# FUNCTIONS
# =====================================================
def generate_key():
    return secrets.token_urlsafe(KEY_LENGTH)

def hash_key(key):
    return hashlib.sha256(key.encode()).hexdigest()

def send_email(to_email, full_name, key):
    body = f"""
Hello {full_name},

Your secure voting key is:

{key}

Please keep this confidential.
Each key can be used only once.

Cast your vote on this form: https://forms.gle/xihoZJJ4W6nZqtRt6

Regards,
Election Commission
EC 2026

Note: This is just for filling the Mock Form only.
"""

    msg = MIMEText(body)

    msg["Subject"] = "Your Voting Key"
    msg["From"] = "EC 2026 <rajatshuvra.saha@niser.ac.in>"
    msg["To"] = to_email

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)

# =====================================================
# LOAD EXCEL SHEETS
# =====================================================
xls = pd.ExcelFile(excel_file)

all_keys = []

# =====================================================
# PROCESS ALL 4 BATCHES
# =====================================================
for sheet in xls.sheet_names:

    print(f"\nProcessing {sheet} ...")

    df = pd.read_excel(excel_file, sheet_name=sheet)

    df.columns = [str(c).strip() for c in df.columns]

    # Detect columns
    name_col = None
    email_col = None

    for c in df.columns:
        low = c.lower()

        if "name" in low:
            name_col = c

        if "email" in low:
            email_col = c

    if name_col is None or email_col is None:
        print(f"Skipped {sheet}")
        continue

    # --------------------------------------------
    # Send mails
    # --------------------------------------------
    for _, row in df.iterrows():

        full_name = str(row[name_col]).strip()
        to_email = str(row[email_col]).strip()

        if to_email == "" or to_email.lower() == "nan":
            continue

        key = generate_key()
        hashed = hash_key(key)

        try:
            send_email(to_email, full_name, key)

            print(f"Sent -> {full_name} | {to_email}")

            all_keys.append({
                "Batch": sheet,
                "Email": to_email,
                "Voting Key": key,
                "SHA256 Hash": hashed,
                "Status": "Sent"
            })

        except Exception as e:

            print(f"Failed -> {to_email} | {e}")

            all_keys.append({
                "Batch": sheet,
                "Email": to_email,
                "Voting Key": key,
                "SHA256 Hash": hashed,
                "Status": f"Failed: {e}"
            })

        time.sleep(DELAY)

# =====================================================
# SAVE MASTER KEY LOG
# =====================================================
out = pd.DataFrame(all_keys)

out.to_excel(
    r"C:\Users\Rajat\Downloads\Voting_Keys_Log.xlsx",
    index=False
)

print("\n✅ All Done.")
print("Saved: Voting_Keys_Log.xlsx")



Processing Sheet1 ...
Sent -> Ranjit Saha | ranjit.saha45@gmail.com
Sent -> Manisha Saha | manishasaha462@gmail.com
Sent -> Rajat Shuvra Saha | rajatshuvra.saha@gmail.com

✅ All Done.
Saved: Voting_Keys_Log.xlsx


### This code is to be run

In [1]:
import pandas as pd
import secrets
import hashlib
import smtplib
import time
import re

from email.mime.text import MIMEText

# =====================================================
# EMAIL LOGIN
# =====================================================
EMAIL = "rajatshuvra.saha@niser.ac.in"
PASSWORD = "jfij ehfa hrjo shbx"

# =====================================================
# EXCEL FILE WITH SHEETS
# =====================================================
excel_file = r"C:\Users\Rajat\Downloads\Testing_my_Names.xlsx"

# =====================================================
# SETTINGS
# =====================================================
KEY_LENGTH = 5
DELAY = 1   # seconds between mails

# =====================================================
# FUNCTIONS
# =====================================================
def generate_key():
    return secrets.token_urlsafe(KEY_LENGTH)

def hash_key(key):
    return hashlib.sha256(key.encode()).hexdigest()


# -----------------------------------------------------
def get_greeting_name(full_name):

    full_name = str(full_name).strip()

    if full_name == "":
        return "Student"

    parts = full_name.split()

    first = parts[0].strip()

    # Single letter or initial
    if len(first) == 1:
        return "Student"

    if re.fullmatch(r"[A-Za-z]\.?", first):
        return "Student"

    return first.title()

# -----------------------------------------------------
# Send email
# -----------------------------------------------------
def send_email(to_email, greeting_name, key):

    body = f"""
Dear {greeting_name},

Your secure voting key is:

{key}

Please keep this confidential.
Each key can be used only once. The code is case-sensitive.

Cast your vote on this form:
https://forms.gle/xihoZJJ4W6nZqtRt6

Regards,
Election Commission
EC 2026

Note: This is just for filling the Mock Form only.
"""

    msg = MIMEText(body)

    msg["Subject"] = "Your Voting Key"
    msg["From"] = "EC 2026 <rajatshuvra.saha@niser.ac.in>"
    msg["To"] = to_email

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)

# =====================================================
# LOAD EXCEL
# =====================================================
xls = pd.ExcelFile(excel_file)

all_keys = []

# =====================================================
# PROCESS ALL SHEETS
# =====================================================
for sheet in xls.sheet_names:

    print(f"\nProcessing {sheet} ...")

    df = pd.read_excel(excel_file, sheet_name=sheet)

    df.columns = [str(c).strip() for c in df.columns]

    # Detect columns
    name_col = None
    email_col = None

    for c in df.columns:
        low = c.lower()

        if "name" in low:
            name_col = c

        if "email" in low:
            email_col = c

    if name_col is None or email_col is None:
        print(f"Skipped {sheet}")
        continue

    # =================================================
    # SEND EMAILS
    # =================================================
    for _, row in df.iterrows():

        full_name = str(row[name_col]).strip()
        to_email = str(row[email_col]).strip()

        if to_email == "" or to_email.lower() == "nan":
            continue

        greeting_name = get_greeting_name(full_name)

        key = generate_key()
        hashed = hash_key(key)

        try:
            send_email(to_email, greeting_name, key)

            print(f"Sent -> Dear {greeting_name} | {to_email}")

            all_keys.append({
                "Batch": sheet,
                "Full Name": full_name,
                "Greeting Used": greeting_name,
                "Email": to_email,
                "Voting Key": key,
                "SHA256 Hash": hashed,
                "Status": "Sent"
            })

        except Exception as e:

            print(f"Failed -> {to_email} | {e}")

            all_keys.append({
                "Batch": sheet,
                "Full Name": full_name,
                "Greeting Used": greeting_name,
                "Email": to_email,
                "Voting Key": key,
                "SHA256 Hash": hashed,
                "Status": f"Failed: {e}"
            })

        time.sleep(DELAY)

# =====================================================
# SAVE LOG
# =====================================================
out = pd.DataFrame(all_keys)

out.to_excel(
    r"C:\Users\Rajat\Downloads\Voting_Keys_Log.xlsx",
    index=False
)

print("\n✅ All Done.")
print("Saved: Voting_Keys_Log.xlsx")


Processing Sheet1 ...
Sent -> Dear Ansh | ansh.droch@niser.ac.in
Sent -> Dear Aysha | ayshainayat.khot@niser.ac.in
Sent -> Dear Jyotirmayee | jyotirmayee.rath@niser.ac.in
Sent -> Dear Sujal | sujal.sinha@niser.ac.in
Sent -> Dear Rajat | rajatshuvra.saha@niser.ac.in
Sent -> Dear Shakya | shakyaratna.wahane@niser.ac.in

✅ All Done.
Saved: Voting_Keys_Log.xlsx


In [ ]:
EMAIL = "rajatshuvra.saha@niser.ac.in"       # your sender email
PASSWORD = "jfij ehfa hrjo shbx"       # Gmail App Password

recipients = [
    "ranjit.saha45@gmail.com",
    "manishasaha462@gmail.com",
    "rajatshuvra.saha@gmail.com"
]

KEY_LENGTH =5


# -----------------------------
# FUNCTIONS
# -----------------------------
def generate_key():
    return secrets.token_urlsafe(KEY_LENGTH)

def hash_key(key):
    return hashlib.sha256(key.encode()).hexdigest()

def send_email(to_email, key):
    msg = MIMEText(f"""
Hello,

Your secure voting key is:

{key}

Please keep this confidential. Each key can be used only once.

Regards,  
Election Commission
""")

    msg['Subject'] = "Your Voting Key"
    msg['From'] = "EC 2026 <rajatshuvra.saha@gmail.com>"
    msg['To'] = to_email

    with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)


# -----------------------------
# MAIN WORKFLOW
# -----------------------------


In [30]:
import secrets
import hashlib
import smtplib
import pandas as pd
import time

from email.mime.text import MIMEText

# =====================================================
# LOGIN
# =====================================================
EMAIL = "rajatshuvra.saha@niser.ac.in"
PASSWORD = "jfij ehfa hrjo shbx"

# =====================================================
# RECIPIENTS
# =====================================================
recipients = [
    "ayshainayat.khot@niser.ac.in",
    "shakyaratna.wahane@niser.ac.in",
    "ansh.droch@niser.ac.in",
    "sujal.sinha@niser.ac.in",
    "jyotirmayee.rath@niser.ac.in",
    "rajatshuvra.saha@niser.ac.in",

]

# =====================================================
# SETTINGS
# =====================================================
KEY_LENGTH = 5
DELAY = 2   # seconds

# =====================================================
# FUNCTIONS
# =====================================================
def generate_key():
    return secrets.token_urlsafe(KEY_LENGTH)

def hash_key(key):
    return hashlib.sha256(key.encode()).hexdigest()

def send_email(to_email, key):

    msg = MIMEText(f"""
Dear Member,

Your secure voting key is:

{key}

Please keep this confidential.
Each key can be used only once.

Regards,
Election Commission
EC 2026

PS: This is just for testing the workflow tomorrow. If you have received this, the code seems to be fine.
""")

    msg["Subject"] = "Your Voting Key"
    msg["From"] = "EC 2026 <rajatshuvra.saha@niser.ac.in>"
    msg["To"] = to_email

    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
        server.login(EMAIL, PASSWORD)
        server.send_message(msg)

# =====================================================
# MAIN WORKFLOW
# =====================================================
records = []

for email in recipients:

    key = generate_key()
    hashed = hash_key(key)

    try:
        send_email(email, key)

        print(f"Sent -> {email}")

        records.append({
            "Email": email,
            "Voting Key": key,
            "SHA256 Hash": hashed,
            "Status": "Sent"
        })

    except Exception as e:

        print(f"Failed -> {email} | {e}")

        records.append({
            "Email": email,
            "Voting Key": key,
            "SHA256 Hash": hashed,
            "Status": f"Failed: {e}"
        })

    time.sleep(DELAY)

# =====================================================
# SAVE LOG
# =====================================================
df = pd.DataFrame(records)

df.to_excel(
    r"C:\Users\Rajat\Downloads\Test_Voting_Key_Log.xlsx",
    index=False
)

print("\n✅ Done.")
print("Saved: Test_Voting_Key_Log.xlsx")

Sent -> ayshainayat.khot@niser.ac.in
Sent -> shakyaratna.wahane@niser.ac.in
Sent -> ansh.droch@niser.ac.in
Sent -> sujal.sinha@niser.ac.in
Sent -> jyotirmayee.rath@niser.ac.in
Sent -> rajatshuvra.saha@niser.ac.in

✅ Done.
Saved: Test_Voting_Key_Log.xlsx


In [10]:
plain_data = []
hash_data = []

for email in recipients:
    key = generate_key()
    h = hash_key(key)

    plain_data.append([email, key])
    hash_data.append([h, "FALSE"])

    print(f"Sending key to {email}...")
    send_email(email, key)


Sending key to ranjit.saha45@gmail.com...
Sending key to manishasaha462@gmail.com...
Sending key to rajatshuvra.saha@gmail.com...


In [8]:
# plaintext (KEEP PRIVATE)
with open("keys_plain.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["email", "key"])
    writer.writerows(plain_data)

# hash (UPLOAD TO GOOGLE SHEET)
with open("keys_hash.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["hash", "used"])
    writer.writerows(hash_data)

print("✅ Done! Keys generated, emailed, and saved.")

✅ Done! Keys generated, emailed, and saved.


In [2]:
import hashlib
import pandas as pd

# =====================================================
# ORIGINAL KEYS
# =====================================================
keys = [
    "2eg4RMc",
    "3E-GIPY",
    "UNRLzN4",
    "KvDu_xg",
    "EhAujak",
    "RkBdtfg"
]

# =====================================================
# GIVEN HASHES
# =====================================================
target_hashes = [
    "b843beab9b568a07e256a231909b1a030d3db8457542857f2315fe8792e5e210",
    "18bf3b95ee4c925c05588ceca071390428a276b71bf919e3d8309e3e09300810",
    "4419ef469f58de8e4e5840ffb19525531cebc1b23b3cc98dd11b8a96f05e805b",
    "850bc5b4d4302d96588dc6122d1e3bca31f347e953c680a905dac71a402773da",
    "c2a4371f2f17e6cbd2daea4922ea4e8b60068a36e4bb1904abd4556b074c2ecd",
    "bb47103b70827ce3d9364f39c47de1dec80b10a1ceda5b67c29e04d114c9c05d"
]

# =====================================================
# HASH + COMPARE
# =====================================================
results = []

for key in keys:

    hashed = hashlib.sha256(key.encode()).hexdigest()

    if hashed in target_hashes:
        status = "MATCH FOUND"
        matched_hash = hashed
    else:
        status = "NO MATCH"
        matched_hash = ""

    results.append({
        "Original Key": key,
        "SHA256 Hash": hashed,
        "Status": status,
        "Matched Hash": matched_hash
    })

# =====================================================
# SHOW RESULTS
# =====================================================
df = pd.DataFrame(results)

print(df.to_string(index=False))

# Optional save
df.to_excel("Hash_Comparison_Result.xlsx", index=False)
print("\nSaved: Hash_Comparison_Result.xlsx")

Original Key                                                      SHA256 Hash      Status                                                     Matched Hash
     2eg4RMc b843beab9b568a07e256a231909b1a030d3db8457542857f2315fe8792e5e210 MATCH FOUND b843beab9b568a07e256a231909b1a030d3db8457542857f2315fe8792e5e210
     3E-GIPY 18bf3b95ee4c925c05588ceca071390428a276b71bf919e3d8309e3e09300810 MATCH FOUND 18bf3b95ee4c925c05588ceca071390428a276b71bf919e3d8309e3e09300810
     UNRLzN4 4419ef469f58de8e4e5840ffb19525531cebc1b23b3cc98dd11b8a96f05e805b MATCH FOUND 4419ef469f58de8e4e5840ffb19525531cebc1b23b3cc98dd11b8a96f05e805b
     KvDu_xg 850bc5b4d4302d96588dc6122d1e3bca31f347e953c680a905dac71a402773da MATCH FOUND 850bc5b4d4302d96588dc6122d1e3bca31f347e953c680a905dac71a402773da
     EhAujak c2a4371f2f17e6cbd2daea4922ea4e8b60068a36e4bb1904abd4556b074c2ecd MATCH FOUND c2a4371f2f17e6cbd2daea4922ea4e8b60068a36e4bb1904abd4556b074c2ecd
     RkBdtfg bb47103b70827ce3d9364f39c47de1dec80b10a1ceda5b67c29e04d11